# MultiVI annotation workflow

One notebook for the Cornell+BCH MultiVI object: Leiden/UMAP, major annotation plots, Cornell ATAC gene activity with SnapATAC2, and annotated `h5mu` export.

In [ ]:
import gzip
import re
import shutil
import warnings
from pathlib import Path

import anndata as ad
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import mudata as mu
import numpy as np
import pandas as pd
import requests
import scanpy as sc
import snapatac2 as snap
from scipy import sparse

warnings.filterwarnings("ignore", category=FutureWarning)
mu.set_options(pull_on_update=False)

BASE_DIR = Path("/Users/mingkewu/Documents/skeletal/mutivi/cornell+bch")
OUT_DIR = BASE_DIR / "annotation_multivi_res0.3_0.7"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_H5MU = BASE_DIR / "mdata.h5mu"
FULL_ANNOTATED_H5MU = OUT_DIR / "mutivi_annotation.h5mu"
FAP_H5MU = OUT_DIR / "fap_res0.3_clusters0_10_17.h5mu"

RESOLUTIONS = [0.3, 0.4, 0.5, 0.6, 0.7]
CLUSTER_KEY = "leiden_multivi_res0.3"
ANNOTATION_KEY = "major_annotation_res0.3"
FAP_CLUSTERS = {"0", "10", "17"}

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=120, fontsize=8)

In [ ]:
CLUSTER_TO_ANNOTATION = {
    "0": "FAP",
    "17": "FAP",
    "10": "FAP",
    "1": "Hybrid FAP",
    "2": "Type I myonuclei",
    "18": "Type I myonuclei",
    "3": "Type II myonuclei",
    "5": "Type II myonuclei",
    "4": "Hybrid myonuclei",
    "6": "Hybrid myonuclei",
    "12": "MuSC",
    "9": "Adipocyte",
    "7": "Endothelial",
    "15": "Lymphatic EC",
    "13": "Pericyte/SMC",
    "8": "Myeloid",
    "14": "HPGDS+ immune",
    "11": "T/NK cells",
    "16": "B cells",
}

ANNOTATION_ORDER = [
    "Type I myonuclei", "Type II myonuclei", "Hybrid myonuclei", "MuSC",
    "FAP", "Hybrid FAP", "Adipocyte", "Endothelial", "Lymphatic EC",
    "Pericyte/SMC", "Myeloid", "HPGDS+ immune", "T/NK cells", "B cells",
]

SHORT_LABELS = {
    "Type I myonuclei": "Type I", "Type II myonuclei": "Type II",
    "Hybrid myonuclei": "Hybrid myo", "MuSC": "MuSC", "FAP": "FAP",
    "Hybrid FAP": "Hybrid FAP", "Adipocyte": "Adipo", "Endothelial": "Endo",
    "Lymphatic EC": "Lymph EC", "Pericyte/SMC": "Peri/SMC",
    "Myeloid": "Myeloid", "HPGDS+ immune": "HPGDS+", "T/NK cells": "T/NK", "B cells": "B",
}

LABEL_OFFSETS = {
    "Type II myonuclei": (-0.45, 0.05),
    "Hybrid myonuclei": (0.62, -0.15),
    "Type I myonuclei": (0.05, -0.45),
    "HPGDS+ immune": (0.15, -0.18),
    "B cells": (0.35, 0.22),
    "Lymphatic EC": (0.25, 0.18),
}

MARKER_GROUPS = {
    "Type I myonuclei": ["MYH7", "TNNT1", "TNNI1", "ATP2A2"],
    "Type II myonuclei": ["MYH1", "MYH2", "TNNT3", "TNNI2", "ATP2A1"],
    "MuSC": ["PAX7", "NCAM1", "CALCR"],
    "FAP": ["PDGFRA", "DCN", "COL1A1", "COL1A2", "COL6A3", "LAMA2"],
    "Adipocyte": ["PLIN1", "PPARG", "ADIPOQ", "GPAM"],
    "Endothelial": ["EMCN", "VWF", "PECAM1", "FLT1"],
    "Lymphatic EC": ["PROX1", "MMRN1", "LYVE1", "PDPN"],
    "Pericyte/SMC": ["PDGFRB", "RGS5", "TAGLN", "MYH11", "CARMN"],
    "Myeloid": ["LRMDA", "MRC1", "CD68", "C1QA", "C1QB", "APOE", "MS4A6A"],
    "HPGDS+ immune": ["HPGDS", "IL18R1"],
    "T/NK cells": ["PTPRC", "SKAP1", "CD247", "CD2", "GNLY", "KLRD1"],
    "B cells": ["BANK1", "EBF1", "BACH2", "PRKCB", "HLA-DRA", "HLA-DRB1"],
}

In [ ]:
def resolution_label(resolution):
    return f"{resolution:.1f}"


def add_major_annotation(adata):
    clusters = adata.obs[CLUSTER_KEY].astype(str)
    missing = sorted(set(clusters.unique()) - set(CLUSTER_TO_ANNOTATION))
    if missing:
        raise ValueError(f"Missing annotation for clusters: {', '.join(missing)}")
    labels = clusters.map(CLUSTER_TO_ANNOTATION)
    adata.obs[ANNOTATION_KEY] = pd.Categorical(
        labels,
        categories=[x for x in ANNOTATION_ORDER if x in set(labels)],
        ordered=True,
    )


def clean_df_for_write(df):
    clean = df.copy()
    clean.index = pd.Index(clean.index.astype(str), name=None)
    for col in clean.columns:
        values = clean[col]
        if isinstance(values.dtype, pd.CategoricalDtype):
            clean[col] = values.astype(str).astype("category")
        elif pd.api.types.is_object_dtype(values) or pd.api.types.is_string_dtype(values):
            clean[col] = values.astype(str)
    return clean


def make_marker_adata(mdata, adata, marker_groups, normalizer_col="nCount_RNA"):
    rna = mdata.mod["rna"]
    var_names = set(map(str, rna.var_names))
    present_groups = {}
    missing = []
    for group, genes in marker_groups.items():
        present = [g for g in genes if g in var_names]
        if present:
            present_groups[group] = present
        missing.extend([g for g in genes if g not in var_names])

    genes = [g for group_genes in present_groups.values() for g in group_genes]
    X = rna[:, genes].X
    X = X.tocsr() if sparse.issparse(X) else sparse.csr_matrix(X)
    totals = adata.obs[normalizer_col].to_numpy(dtype=np.float64)
    scale = np.divide(1e4, totals, out=np.zeros_like(totals), where=totals > 0)
    X = X.multiply(scale[:, None]).tocsr()
    X.data = np.log1p(X.data)

    marker_adata = ad.AnnData(
        X=X,
        obs=adata.obs.copy(),
        var=pd.DataFrame(index=pd.Index(genes, name=None)),
    )
    if missing:
        print("Skipped missing RNA marker genes:", ", ".join(missing))
    return marker_adata, present_groups

In [ ]:
# Load MultiVI latent space and compute neighbors / UMAP / Leiden.
mdata = mu.read_h5mu(SOURCE_H5MU, backed="r")
adata = ad.AnnData(X=np.asarray(mdata.obsm["X_multivi"], dtype=np.float32), obs=clean_df_for_write(mdata.obs))
adata.obsm["X_multivi"] = adata.X.copy()

sc.pp.neighbors(adata, use_rep="X_multivi", n_neighbors=15, metric="euclidean", random_state=0)
sc.tl.umap(adata, min_dist=0.2, random_state=0)

for res in RESOLUTIONS:
    key = f"leiden_multivi_res{resolution_label(res)}"
    sc.tl.leiden(
        adata,
        resolution=res,
        key_added=key,
        random_state=0,
        flavor="igraph",
        n_iterations=2,
        directed=False,
    )
    sc.pl.umap(adata, color=key, legend_loc="on data", frameon=False, size=1.5, show=False)
    plt.savefig(OUT_DIR / f"umap_{key}.pdf", bbox_inches="tight")
    plt.close()

In [ ]:
# Marker dotplots for Leiden resolutions 0.3-0.7.
marker_adata, present_marker_groups = make_marker_adata(mdata, adata, MARKER_GROUPS)

for res in RESOLUTIONS:
    key = f"leiden_multivi_res{resolution_label(res)}"
    n_genes = sum(len(v) for v in present_marker_groups.values())
    n_clusters = marker_adata.obs[key].nunique()
    dp = sc.pl.dotplot(
        marker_adata,
        var_names=present_marker_groups,
        groupby=key,
        use_raw=False,
        standard_scale="var",
        figsize=(max(12, n_genes * 0.34), max(4, n_clusters * 0.28)),
        show=False,
        return_fig=True,
    )
    dp.savefig(OUT_DIR / f"dotplot_major_markers_{key}.pdf")
    plt.close("all")

In [ ]:
# Add merged major annotation and draw merged UMAP + dotplot.
add_major_annotation(adata)

coords = adata.obsm["X_umap"]
labels = adata.obs[ANNOTATION_KEY]
colors = plt.get_cmap("tab20").colors
fig, ax = plt.subplots(figsize=(6, 5.8))
for i, label in enumerate(labels.cat.categories):
    mask = labels == label
    ax.scatter(coords[mask, 0], coords[mask, 1], s=1.2, c=[colors[i % len(colors)]], linewidths=0, alpha=0.8, rasterized=True)
for label in labels.cat.categories:
    mask = labels == label
    x, y = np.median(coords[mask, 0]), np.median(coords[mask, 1])
    dx, dy = LABEL_OFFSETS.get(label, (0, 0))
    ax.text(x + dx, y + dy, SHORT_LABELS.get(label, label), ha="center", va="center", fontsize=8, fontweight="bold",
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.65, "pad": 1.2})
ax.set(title="major_annotation_res0.3", xticks=[], yticks=[], xlabel="", ylabel="")
for spine in ax.spines.values():
    spine.set_visible(False)
fig.tight_layout()
fig.savefig(OUT_DIR / "umap_major_annotation_res0.3.pdf", bbox_inches="tight")
plt.close(fig)

marker_adata.obs[ANNOTATION_KEY] = adata.obs[ANNOTATION_KEY].copy()
dp = sc.pl.dotplot(
    marker_adata,
    var_names=present_marker_groups,
    groupby=ANNOTATION_KEY,
    categories_order=[x for x in ANNOTATION_ORDER if x in set(adata.obs[ANNOTATION_KEY].astype(str))],
    use_raw=False,
    standard_scale="var",
    figsize=(18, 5.5),
    show=False,
    return_fig=True,
)
dp.savefig(OUT_DIR / "dotplot_major_annotation_res0.3.pdf")
plt.close("all")

In [ ]:
# Save annotated full h5mu and FAP-only h5mu.
if FULL_ANNOTATED_H5MU.exists():
    FULL_ANNOTATED_H5MU.unlink()
shutil.copy2(SOURCE_H5MU, FULL_ANNOTATED_H5MU)

full = mu.read_h5mu(FULL_ANNOTATED_H5MU, backed="r+")
full.obs = clean_df_for_write(full.obs)
full.var = clean_df_for_write(full.var)
for mod in ["rna", "atac"]:
    full.mod[mod].obs = clean_df_for_write(full.mod[mod].obs)
    full.mod[mod].var = clean_df_for_write(full.mod[mod].var)

if not np.array_equal(full.obs_names.astype(str), adata.obs_names.astype(str)):
    raise ValueError("mdata and annotation object obs_names are not aligned")

anno_cols = [f"leiden_multivi_res{resolution_label(res)}" for res in RESOLUTIONS] + [ANNOTATION_KEY]
for col in anno_cols:
    values = adata.obs[col].copy()
    full.obs[col] = values
    full.mod["rna"].obs[col] = values
    full.mod["atac"].obs[col] = values
full.obsm["X_umap"] = np.asarray(adata.obsm["X_umap"], dtype=np.float32)
full.write()

full = mu.read_h5mu(FULL_ANNOTATED_H5MU, backed="r")
fap_cells = full.obs_names[full.obs[CLUSTER_KEY].astype(str).isin(FAP_CLUSTERS)].astype(str).tolist()
if FAP_H5MU.exists():
    FAP_H5MU.unlink()
full[fap_cells, :].copy(FAP_H5MU)
print(f"Wrote {FULL_ANNOTATED_H5MU}")
print(f"Wrote {FAP_H5MU} with {len(fap_cells):,} FAP cells")

In [ ]:
# SnapATAC2 gene activity dotplot on Cornell observed ATAC cells only.
GENCODE_URL = "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_50/gencode.v50.annotation.gtf.gz"
GENCODE_GTF = Path("/tmp/gencode.v50.annotation.gtf.gz")
MARKER_GTF = Path("/tmp/gencode.v50.multivi_marker_genes.gtf.gz")
GENE_NAME_RE = re.compile(r'gene_name "([^"]+)"')

if not GENCODE_GTF.exists() or GENCODE_GTF.stat().st_size < 10_000_000:
    with requests.get(GENCODE_URL, stream=True, timeout=60) as r:
        r.raise_for_status()
        with GENCODE_GTF.open("wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

marker_genes = []
seen = set()
for genes in MARKER_GROUPS.values():
    for gene in genes:
        if gene not in seen:
            seen.add(gene)
            marker_genes.append(gene)

with gzip.open(GENCODE_GTF, "rt") as src, gzip.open(MARKER_GTF, "wt") as out:
    for line in src:
        if line.startswith("#"):
            continue
        match = GENE_NAME_RE.search(line)
        if match and match.group(1) in seen:
            out.write(line)

full = mu.read_h5mu(FULL_ANNOTATED_H5MU, backed="r")
atac = full.mod["atac"]
cornell_mask = full.obs["has_atac"].astype(bool).to_numpy()
cornell_idx = np.flatnonzero(cornell_mask)

atac_cornell = atac[cornell_idx, :].to_memory()
atac_cornell.obs = clean_df_for_write(full.obs.iloc[cornell_idx])
atac_cornell.var = pd.DataFrame(index=atac_cornell.var_names.astype(str))
atac_cornell.X = atac_cornell.X.tocsr().astype(np.uint32) if sparse.issparse(atac_cornell.X) else np.asarray(atac_cornell.X, dtype=np.uint32)

gene_activity = snap.pp.make_gene_matrix(
    atac_cornell,
    gene_anno=MARKER_GTF,
    use_x=True,
    inplace=False,
    upstream=2000,
    downstream=0,
    include_gene_body=True,
    chunk_size=500,
)

gene_activity.X = gene_activity.X.tocsr() if sparse.issparse(gene_activity.X) else sparse.csr_matrix(gene_activity.X)
totals = atac_cornell.obs["nCount_ATAC"].to_numpy(dtype=np.float64)
scale = np.divide(1e4, totals, out=np.zeros_like(totals), where=totals > 0)
gene_activity.X = gene_activity.X.multiply(scale[:, None]).tocsr()
gene_activity.X.data = np.log1p(gene_activity.X.data)
gene_activity.obs = atac_cornell.obs[[ANNOTATION_KEY, CLUSTER_KEY, "orig.ident", "cohort"]].copy()

available = set(map(str, gene_activity.var_names))
nonzero = np.asarray(gene_activity.X.sum(axis=0)).ravel() > 0
nonzero_genes = set(gene_activity.var_names[nonzero].astype(str))
activity_marker_groups = {
    group: [gene for gene in genes if gene in available and gene in nonzero_genes]
    for group, genes in MARKER_GROUPS.items()
}
activity_marker_groups = {group: genes for group, genes in activity_marker_groups.items() if genes}

dp = sc.pl.dotplot(
    gene_activity,
    var_names=activity_marker_groups,
    groupby=ANNOTATION_KEY,
    categories_order=[x for x in ANNOTATION_ORDER if x in set(gene_activity.obs[ANNOTATION_KEY].astype(str))],
    use_raw=False,
    standard_scale="var",
    colorbar_title="Mean gene activity\nin group",
    figsize=(18, 5.5),
    show=False,
    return_fig=True,
)
dp.savefig(OUT_DIR / "gene_activity_dotplot_major_annotation_res0.3_cornell_atac.pdf")
plt.close("all")

In [ ]:
# Quick checks.
full = mu.read_h5mu(FULL_ANNOTATED_H5MU, backed="r")
fap = mu.read_h5mu(FAP_H5MU, backed="r")
print(full)
print(fap)
print(full.obs[ANNOTATION_KEY].value_counts())
print(fap.obs["has_atac"].value_counts())